# RAG SUNAT — Demo final (≈7 min)

Pipeline: **carga → chunking → embeddings → FAISS HNSW → BM25 → híbrido → reranker → Qwen → citas → grounding → evaluación**.

Este notebook **no** recalcula ingesta ni embeddings del corpus: reutiliza reportes JSON y, en la demo en vivo, un **índice FAISS guardado** + BM25 reconstruido desde metadatos.

**Antes de presentar en Colab:** sube a `data/processed/` los reportes (`*.json`) y la carpeta **`faiss_saved/`** con `faiss_hnsw.index` + `faiss_hnsw_metadata.json` (genéralos en local con `python scripts/test_faiss_hnsw.py --index-dir data/processed/faiss_saved`).


## 1. Setup

Clona el repositorio (Colab), instala dependencias y fija la raíz del proyecto.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(5):
        if (p / "requirements.txt").is_file() and (p / "src").is_dir():
            return p
        p = p.parent
    raise FileNotFoundError("No se encontró la raíz del repo (requirements.txt + src/).")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/junvegu/rag-mds2025-ia-proyectofinal.git"
CLONE_DIR = Path("/content/rag-mds2025-ia-proyectofinal")

if IN_COLAB:
    if not CLONE_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)
    os.chdir(CLONE_DIR)
else:
    pass  # ejecutar con cwd = raíz del repo, o desde notebooks/

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ["RAG_PROJECT_ROOT"] = str(PROJECT_ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.config.runtime_env import apply_darwin_openmp_mitigations

apply_darwin_openmp_mitigations()
print("OK — PROJECT_ROOT =", PROJECT_ROOT)


## 2. Verificación de artefactos

Comprueba que los reportes de pipeline y evaluación estén disponibles (sube `data/processed/` si falta).


In [ ]:
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"
REQUIRED = [
    "ingestion_report.json",
    "chunking_report.json",
    "embeddings_report.json",
    "faiss_report.json",
    "hybrid_retrieval_report.json",
    "reranker_report.json",
    "generation_report.json",
    "evaluation_report.json",
]

for name in REQUIRED:
    p = PROCESSED / name
    status = "OK" if p.is_file() else "FALTA"
    print(f"  [{status}] {name}")


## 3. Resumen final del sistema

Métricas agregadas y conclusión automática desde `evaluation_report.json`.


In [ ]:
import json
from pathlib import Path

eval_path = PROCESSED / "evaluation_report.json"
if not eval_path.is_file():
    print("No hay evaluation_report.json — omite esta sección o súbelo a data/processed/")
else:
    rep = json.loads(eval_path.read_text(encoding="utf-8"))
    gm = rep.get("global_metrics") or {}
    con = rep.get("conclusion") or {}
    print("═" * 56)
    print(" DASHBOARD — Evaluación automática")
    print("═" * 56)
    print(f"  Conclusión      : {con.get('verdict_es', 'N/D')}")
    print(f"  Detalle         : {con.get('summary_es', '')}")
    print(f"  avg_rouge_1     : {gm.get('avg_rouge_1', 'N/D')}")
    print(f"  avg_rouge_l     : {gm.get('avg_rouge_l', 'N/D')}")
    print(f"  avg_grounding   : {gm.get('avg_grounding_score', 'N/D')}")
    print(f"  flags alucinación: {gm.get('total_hallucination_flags', 'N/D')}")
    print("═" * 56)


## 4. Ejemplos de respuestas

Dos ítems del evaluation set con trazabilidad legible.


In [ ]:
eval_path = PROJECT_ROOT / "data" / "processed" / "evaluation_report.json"
if not eval_path.is_file():
    print("Sin evaluation_report.json.")
else:
    rep = json.loads(eval_path.read_text(encoding="utf-8"))
    items = rep.get("by_question") or []
    for i, row in enumerate(items[:2], 1):
        print("\n" + "─" * 56)
        print(f" EJEMPLO {i} — {row.get('question_id', '')}")
        print("─" * 56)
        print("Pregunta:\n", row.get("question", ""))
        print("\nRespuesta (generada):\n", (row.get("generated_answer") or "")[:1200], "…" if len(row.get("generated_answer") or "") > 1200 else "")
        print("\n  grounding_score :", row.get("grounding_score"))
        print("  rouge_l         :", row.get("rouge_l"))
        print("  sources_used    :")
        for s in row.get("sources_used") or []:
            print("   •", s)


## 5. Demo en vivo (1 pregunta)

Carga **solo** el índice FAISS + metadatos; reconstruye BM25 en RAM; ejecuta híbrido → rerank → Qwen → citas. **No** re-ingesta ni re-embedder del corpus.


In [ ]:
from src.interfaces.sunat_faiss_runtime import answer_from_saved_faiss

FAISS_DIR = PROJECT_ROOT / "data" / "processed" / "faiss_saved"
DEMO_QUESTION = "¿Cómo se calcula el impuesto a la renta de quinta categoría?"

need = [FAISS_DIR / "faiss_hnsw.index", FAISS_DIR / "faiss_hnsw_metadata.json"]
if not all(p.is_file() for p in need):
    print("FALTA índice guardado. En local ejecuta:")
    print("  python scripts/test_faiss_hnsw.py --index-dir data/processed/faiss_saved")
    print("y sube la carpeta faiss_saved/ a Colab bajo data/processed/")
else:
    ans = answer_from_saved_faiss(DEMO_QUESTION, faiss_dir=FAISS_DIR, top_k=5, hybrid_pool=10)
    print("Pregunta:\n", DEMO_QUESTION)
    print("\n--- Respuesta ---\n", ans.text[:2000], "…" if len(ans.text) > 2000 else "")
    print("\n  grounding_score    :", ans.grounding_score)
    print("  hallucination_flag :", ans.hallucination_flag)
    print("\n--- Primeras 3 citas (oración → fuente → página) ---")
    for j, c in enumerate((ans.citations or [])[:3], 1):
        print(f"\n[{j}] {c.sentence[:220]}{'…' if len(c.sentence) > 220 else ''}")
        print(f"    source: {c.source}")
        print(f"    page : {c.page}")


## 6. Resumen ejecutivo

Tabla única para cierre (README / slides); combina reportes cuando existan.


In [ ]:
import json
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"

def load_json(path: Path):
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

ing = load_json(PROCESSED / "ingestion_report.json")
chk = load_json(PROCESSED / "chunking_report.json")
fax = load_json(PROCESSED / "faiss_report.json")
evl = load_json(PROCESSED / "evaluation_report.json")

ing_r = (ing or {}).get("report") or {}
chk_r = (chk or {}).get("report") or {}
fax_r = (fax or {}).get("report") or {}
gm = (evl or {}).get("global_metrics") or {}
ex = (evl or {}).get("executive_summary") or {}
models = (ex.get("models_and_index") or {})

resumen = {
    "fuentes": ing_r.get("total_sources"),
    "documentos": ing_r.get("total_documents") or chk_r.get("input_documents"),
    "chunks": chk_r.get("generated_chunks") or fax_r.get("total_chunks"),
    "modelo_embeddings": (evl or {}).get("embedding_model") or models.get("embedding_model"),
    "indice": "FAISS HNSW",
    "retrieval": "Hybrid + Reranker",
    "LLM": ((evl or {}).get("generation_model") or models.get("generation_model") or "Qwen"),
    "grounding_promedio": gm.get("avg_grounding_score"),
    "rouge_1": gm.get("avg_rouge_1"),
    "rouge_l": gm.get("avg_rouge_l"),
}

print(json.dumps(resumen, ensure_ascii=False, indent=2))
